## OpenAI com skill prompts

Esta secao usa os prompts da skill `ecossistema-agentes-prompts` para orquestrador, especialistas e avaliador.

In [1]:
import os
import re
import json
import sys
import time
from typing import Dict, List, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI
from dotenv import load_dotenv
from mnt.skills.agente_mysql.helpers import MySQLAgent
import logging

load_dotenv()

# Logger do Maestro — troque para logging.DEBUG para ver detalhes de execução
logging.basicConfig(
    level=logging.DEBUG, # WARNING,
    format="%(asctime)s [%(name)s] %(levelname)s: %(message)s"
)
logger = logging.getLogger("maestro")

sys.path.insert(0, os.path.abspath("."))

In [2]:
# Caminho base das skills (pode ser absoluto ou relativo ao notebook)
SKILLS_DIR = "mnt/skills"
SKILL_FILE = "SKILL.md"

def _parse_frontmatter(content: str) -> Tuple[Dict, str]:
    """Extrai frontmatter YAML (entre ---) e retorna (metadados, resto do conteúdo)."""
    meta = {}
    body = content
    match = re.match(r"^---\s*\n(.*?)\n---\s*\n(.*)", content, re.DOTALL)
    if match:
        fm, body = match.group(1), match.group(2)
        for line in fm.split("\n"):
            if ":" in line:
                key, val = line.split(":", 1)
                meta[key.strip()] = val.strip().strip("'\"").replace("\n", " ").strip()
    return meta, body.strip()

def load_skills(base_dir: str) -> List[Dict]:
    """
    Carrega todas as skills a partir de base_dir.
    Cada subpasta contendo SKILL.md vira uma skill.
    Retorna lista de dicts: id, name, description, content (texto completo).
    """
    skills = []
    base = os.path.abspath(base_dir)
    if not os.path.isdir(base):
        return skills
    for entry in sorted(os.listdir(base)):
        path = os.path.join(base, entry)
        skill_file = os.path.join(path, SKILL_FILE)
        if os.path.isdir(path) and os.path.isfile(skill_file):
            with open(skill_file, "r", encoding="utf-8") as f:
                raw = f.read()
            meta, body = _parse_frontmatter(raw)
            skill_id = entry
            name = meta.get("name", skill_id)
            description = meta.get("description", "")
            skills.append({
                "id": skill_id,
                "name": name,
                "description": description,
                "model": meta.get("model"),   # modelo definido no SKILL.md
                "content": raw,
                "body": body,
            })
    return skills

def get_skill_by_id(skills: List[Dict], skill_id: str) -> Optional[Dict]:
    """Retorna a skill com o id dado ou None."""
    for s in skills:
        if s["id"] == skill_id:
            return s
    return None

# Carrega todas as skills
skills_loaded = load_skills(SKILLS_DIR)
print(f"Skills carregadas: {len(skills_loaded)}")
for s in skills_loaded:
    print(f"  - {s['id']}: {s.get('name', s['id'])}")
    # print(s['content'])

Skills carregadas: 13
  - agente-analise-concessionaria: agente-analise-concessionaria
  - agente-analise-os: agente-analise-os
  - agente-cientifico: agente-cientifico
  - agente-dados: agente-dados
  - agente-financeiro: agente-financeiro
  - agente-juridico: agente-juridico
  - agente-mercado: agente-mercado
  - agente-mysql: agente-mysql
  - agente-negocios: agente-negocios
  - agente-saude: agente-saude
  - agente-tecnico: agente-tecnico
  - avaliador-coerencia: avaliador-coerencia
  - maestro: maestro


In [3]:
## Defini client e o modelo e carregar as skills
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
model = os.environ.get("MODELO_DEFAULT")
skills = load_skills("mnt/skills")
_t = os.environ.get("TIME_INTERVAL_AGENTS")
time_interval_agents = float(_t) if _t else 2

In [4]:
def chat_with_skill(
    client: OpenAI,
    skill_id: str,
    user_message: str,
    model: str,
    skills_list: Optional[List[Dict]] = None,
) -> str:
    """
    Envia uma mensagem ao modelo OpenAI usando o conteúdo da skill como system prompt.
    Usa o modelo definido no SKILL.md (campo model:); fallback para o parâmetro model.
    """
    skills = skills_list if skills_list is not None else skills_loaded
    skill = get_skill_by_id(skills, skill_id)
    if not skill:
        return f"Erro: skill '{skill_id}' não encontrada."
    modelo = skill.get("model") or model
    resp = client.chat.completions.create(
        model=modelo,
        messages=[
            {"role": "system", "content": skill["content"]},
            {"role": "user", "content": user_message},
        ],
    )
    return resp.choices[0].message.content or ""

# Exemplo:
# client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
# model = os.environ.get("MODELO_DEFAULT")
# resposta = chat_with_skill(client, "agente-tecnico", "Como implementar um RAG com embeddings?", model)
# print(resposta)

In [5]:
def extrair_json(texto: str) -> Optional[Dict]:
    """Extrai JSON do texto retornado pelo modelo (pode vir em bloco ```json ... ```)."""
    if not texto or not texto.strip():
        return None
    s = texto.strip()
    # Remove bloco markdown de código
    if "```" in s:
        match = re.search(r"```(?:json)?\s*([\s\S]*?)```", s)
        if match:
            s = match.group(1).strip()
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        return None

def invocar_agente_maestro(
    client: OpenAI,
    skill_id: str,
    payload_maestro: Dict,
    model: str,
    skills_list: Optional[List[Dict]] = None,
) -> str:
    """
    Invoca um agente em modo Maestro; o modelo deve retornar JSON no Formato de Retorno da skill.
    Usa o modelo definido no SKILL.md (campo model:); fallback para o parâmetro model.
    Retorna o texto bruto da resposta (parse com extrair_json depois).
    """
    skills = skills_list if skills_list is not None else skills_loaded
    skill = get_skill_by_id(skills, skill_id)
    if not skill:
        return ""
    modelo = skill.get("model") or model
    user = (
        json.dumps(payload_maestro, ensure_ascii=False)
        + "\n\nVocê está sendo invocado pelo Maestro. Responda APENAS com um único JSON válido no Formato de Retorno da skill "
        "(agente_id, agente_nome, pode_responder, justificativa_viabilidade, resposta, scores, limitacoes_da_resposta, aspectos_para_outros_agentes). "
        "Sem texto antes ou depois."
    )
    resp = client.chat.completions.create(
        model=modelo,
        messages=[
            {"role": "system", "content": skill["content"]},
            {"role": "user", "content": user},
        ],
        response_format={"type": "json_object"},
    )
    return (resp.choices[0].message.content or "").strip()

In [6]:
# Definição inline removida — use:
from app.services.maestro_fluxo import executar_fluxo_maestro


In [7]:
# PERGUNTA_TESTE = (
#     "Quais são as melhores práticas para uma startup de SaaS precificar o produto "
#     "e quais ferramentas técnicas (APIs, billing) ajudam a implementar isso?"
# )

# resultado = executar_fluxo_maestro(client=client, pergunta=PERGUNTA_TESTE, model=model, agentes=["agente-tecnico", "agente-financeiro"], verbose=True)
# print(resultado["entrega_final"])

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# CÉLULA ÚNICA — Pergunta ao Maestro e ele busca 10 linhas da tabela
# ─────────────────────────────────────────────────────────────────────────────

# ── 1. Pergunta em linguagem natural ─────────────────────────────────────────

# PERGUNTA = "Me mostre 10 linhas da tabela de servicos"

# # ── 2. Maestro identifica qual tabela buscar ──────────────────────────────────

# skill_maestro = get_skill_by_id(skills, "maestro")

# resp = client.chat.completions.create(
#     model=model,
#     messages=[
#         {"role": "system", "content": skill_maestro["content"]},
#         {"role": "user",   "content": (
#             f"{PERGUNTA}\n\n"
#             "Retorne APENAS um JSON com a chave 'tabela' contendo o nome da tabela "
#             "que o usuário quer consultar. Ex: {\"tabela\": \"vendas\"}"
#         )},
#     ],
#     response_format={"type": "json_object"},
# )

# tabela = extrair_json(resp.choices[0].message.content).get("tabela", "")
# print(f"[MAESTRO] Tabela identificada: '{tabela}'")

# # ── 3. agente-mysql busca as 10 linhas ───────────────────────────────────────

# agent = MySQLAgent(
#     host    = os.environ.get("MYSQL_HOST"),
#     banco   = os.environ.get("MYSQL_DATABASE"),
#     usuario = os.environ.get("MYSQL_USER"),
#     senha   = os.environ.get("MYSQL_PASSWORD"),
# )

# resultado = agent.carregar_tabela(tabela, limite=10, verbose=False)

# if not resultado["sucesso"]:
#     print(f"[ERRO] {resultado['erro']}")
# else:
#     df = resultado["dataframe"]
#     print(f"\n✅ Tabela '{tabela}' — 10 primeiras linhas:\n")
#     print(df.to_string(index=False))

In [9]:
# # ─────────────────────────────────────────────────────────────────────────────
# # CÉLULA: MySQL → agente-dados → executa código Pandas
# # ─────────────────────────────────────────────────────────────────────────────

# # ── Configuração ──────────────────────────────────────────────────────────────

# PERGUNTA   = "Quais os 5 serviços com maior custo_fixo e estão ativos?"
# TABELA     = "servicos"

# resultado = executar_fluxo_maestro(
#     client=client,
#     pergunta=PERGUNTA,
#     model=model,
#     agentes=["agente-dados"],
#     verbose=True,
#     mysql_host=os.environ.get("MYSQL_HOST"),
#     mysql_porta=int(os.environ.get("MYSQL_PORT", "3306")),
#     mysql_usuario=os.environ.get("MYSQL_USER"),
#     mysql_senha=os.environ.get("MYSQL_PASSWORD"),
#     mysql_banco=os.environ.get("MYSQL_DATABASE"),
#     mysql_tabela=TABELA,
#     mysql_limite=50_000,
#     mysql_filtro_where="",
#     mysql_injetar_namespace=globals(),
# )

# resp = next((r for r in resultado["respostas_agentes"] if r.get("agente_id") == "agente-dados"), {})

# print(f"\n[agente-dados] pode_responder: {resp.get('pode_responder')}")
# print(f"[agente-dados] score_final:    {resp.get('scores', {}).get('score_final')}")
# print(f"[agente-dados] {resp.get('justificativa_viabilidade')}")

# # ── Executa o código Pandas gerado ───────────────────────────────────────────

# codigo = resp.get("codigo_pandas", "")

# if not codigo:
#     print("\n[AVISO] agente-dados não retornou codigo_pandas.")
#     print("Resposta:", resp.get("resposta"))
# else:
#     print(f"\n{'─'*60}")
#     print("CÓDIGO GERADO PELO agente-dados:")
#     print('─'*60)
#     print(codigo)
#     print('─'*60)
#     print("\nRESULTADO:\n")
#     exec(codigo)   # executa no namespace atual — df_servicos já está disponível

In [10]:
# =============================================================================
# Auxiliar: capturar schema das tabelas para gerar schema.json
# =============================================================================
from sqlalchemy import create_engine, text
import json

engine = create_engine(
    f"mysql+pymysql://{os.environ['MYSQL_USER']}:{os.environ['MYSQL_PASSWORD']}"
    f"@{os.environ['MYSQL_HOST']}:{os.environ.get('MYSQL_PORT','3306')}"
    f"/{os.environ['MYSQL_DATABASE']}?charset=utf8mb4",
    pool_pre_ping=True,
)

tabelas = ["os", "os_servicos", "servicos", "concessionarias", "funcionarios", "caixas"]
schema = {}

with engine.connect() as conn:
    for tabela in tabelas:
        rows = conn.execute(text(f"DESCRIBE `{tabela}`")).fetchall()
        schema[tabela] = [
            {"coluna": r[0], "tipo": str(r[1]), "nullable": r[2], "key": r[3], "default": str(r[4]) if r[4] else None}
            for r in rows
        ]
        print(f"\n{'='*50}")
        print(f"TABELA: {tabela} ({len(rows)} colunas)")
        print(f"{'='*50}")
        for r in rows:
            print(f"  {r[0]:<30} {str(r[1]):<20} {'NULL' if r[2]=='YES' else 'NOT NULL':<10} {r[3] or ''}")

print(f"\n\nSchema capturado: {len(schema)} tabelas")
for t, cols in schema.items():
    print(f"  {t}: {len(cols)} colunas")


TABELA: os (81 colunas)
  id                             int unsigned         NOT NULL   PRI
  uuid                           char(36)             NOT NULL   UNI
  os_concessionaria              int unsigned         NOT NULL   
  tipo_atendimento               enum('PRESENCIAL','TELEFONE','WHATSAPP') NULL       
  valor_bruto                    decimal(8,2) unsigned NOT NULL   
  valor_liquido                  decimal(8,2) unsigned NOT NULL   
  retencao_iss                   tinyint(1)           NOT NULL   
  paga                           tinyint(1)           NOT NULL   MUL
  data_pagamento                 datetime             NULL       MUL
  observacao_pagamento           text                 NULL       
  usuario_pagamento_id           int unsigned         NULL       MUL
  fechada                        tinyint(1)           NOT NULL   MUL
  data_fechamento                datetime             NULL       
  finalizada                     tinyint(1)           NOT NULL   
  data_fina

In [11]:
# =============================================================================
# Teste do agente-analise-os (FASE 1 + FASE 2) — apenas texto, sem gráficos
# =============================================================================
# Colunas prefixadas para evitar duplicate keys:
#   os.*          -> sem prefixo (tabela principal, 81 colunas)
#   os_servicos   -> oss_ (valores, descontos, status do servico)
#   servicos      -> ser_ (catalogo) + alias servico_nome
#   concessionarias -> con_ (cadastro) + alias concessionaria_nome
#   funcionarios  -> func_ (vendedor) + alias vendedor_nome
#   caixas        -> NAO JOINADA (EXISTS subquery -> os_paga)
#   qtd_servicos  -> subquery COUNT (cross-selling)

# import pandas as pd

# skills = load_skills("mnt/skills")
# print(f"Skills carregadas: {len(skills)}")
# for s in skills:
#     print(f"  - {s['id']}: {s.get('name', s['id'])}")

# PERGUNTA_TESTE_OS = "Análise semanal completa de Ordens de Serviço"

# resultado_os = executar_fluxo_maestro(
#     client=client,
#     pergunta=PERGUNTA_TESTE_OS,
#     model=model,
#     agentes=["agente-analise-os"],
#     agentes_dataframe=["agente-analise-os"],
#     verbose=True,
#     mysql_host=os.environ.get("MYSQL_HOST"),
#     mysql_porta=int(os.environ.get("MYSQL_PORT", "3306")),
#     mysql_usuario=os.environ.get("MYSQL_USER"),
#     mysql_senha=os.environ.get("MYSQL_PASSWORD"),
#     mysql_banco=os.environ.get("MYSQL_DATABASE"),
#     mysql_tabelas=[
#         {"tabela": "os", "alias": "os", "colunas": [
#             "`os`.*",
#             "EXISTS(SELECT 1 FROM caixas cx WHERE cx.os_id = os.id AND cx.cancelado = 0 AND cx.deleted_at IS NULL) AS os_paga",
#             "(SELECT COUNT(*) FROM os_servicos oss2 WHERE oss2.os_id = os.id) AS qtd_servicos",
#         ]},
#         {"tabela": "os_servicos", "alias": "oss", "fk": "os.id = oss.os_id", "colunas": [
#             "oss.id AS oss_id",
#             "oss.codigo AS oss_codigo",
#             "oss.valor_venda AS oss_valor_venda",
#             "oss.valor_original AS oss_valor_original",
#             "oss.desconto_supervisao AS oss_desconto_supervisao",
#             "oss.desconto_migracao_cortesia AS oss_desconto_migracao_cortesia",
#             "oss.desconto_avista AS oss_desconto_avista",
#             "oss.valor_venda_real AS oss_valor_venda_real",
#             "oss.desconto_bonus AS oss_desconto_bonus",
#             "oss.fechado AS oss_fechado",
#             "oss.cancelado AS oss_cancelado",
#             "oss.servico_id AS oss_servico_id",
#             "oss.os_tipo_id AS oss_os_tipo_id",
#             "oss.combo_id AS oss_combo_id",
#             "oss.concessionaria_execucao_id AS oss_concessionaria_execucao_id",
#             "oss.created_at AS oss_created_at",
#             "oss.deleted_at AS oss_deleted_at",
#         ]},
#         {"tabela": "servicos", "alias": "ser", "fk": "oss.servico_id = ser.id", "colunas": [
#             "ser.id AS ser_id",
#             "ser.nome AS servico_nome",
#             "ser.custo_fixo AS ser_custo_fixo",
#             "ser.grupo_servico_id AS ser_grupo_servico_id",
#             "ser.subgrupo_servico_id AS ser_subgrupo_servico_id",
#             "ser.servico_categoria_id AS ser_servico_categoria_id",
#         ]},
#         {"tabela": "concessionarias", "alias": "con", "fk": "os.concessionaria_id = con.id", "colunas": [
#             "con.id AS con_id",
#             "con.nome AS concessionaria_nome",
#             "con.uf AS con_uf",
#             "con.localidade AS con_localidade",
#             "con.cluster_id AS con_cluster_id",
#             "con.business_unit_id AS con_business_unit_id",
#             "con.gerente_nome AS con_gerente_nome",
#         ]},
#         {"tabela": "funcionarios", "alias": "func", "fk": "os.vendedor_id = func.id", "colunas": [
#             "func.id AS func_id",
#             "func.nome AS vendedor_nome",
#             "func.terceiros AS func_terceiros",
#             "func.funcionario_situacao_id AS func_situacao_id",
#         ]},
#     ],
#     mysql_limite=50_000,
#     mysql_filtro_where="os.deleted_at IS NULL AND os.created_at >= '2023-01-01'",
#     mysql_injetar_namespace=globals(),
# )

# # Pós-processamento: normalizar datas (remove timezone se houver)
# df_var = resultado_os.get("df_variavel")
# if df_var and df_var in globals():
#     df_ref = globals()[df_var]
#     for col in ["created_at", "updated_at", "oss_created_at"]:
#         if col in df_ref.columns:
#             df_ref[col] = pd.to_datetime(df_ref[col], errors="coerce").dt.tz_localize(None)
#     print(f"[POS] Shape: {df_ref.shape}")
#     print(f"[POS] Colunas ({len(df_ref.columns)}): {list(df_ref.columns)}")
#     dups = [c for c in df_ref.columns if df_ref.columns.tolist().count(c) > 1]
#     if dups:
#         print(f"[POS] ALERTA: colunas duplicadas: {set(dups)}")
#     else:
#         print("[POS] OK: sem colunas duplicadas")

# # Verificar estrutura da resposta
# resp_agente = resultado_os["respostas_agentes"][0]
# print("\n" + "=" * 60)
# print("FASE 1: pode_responder:", resp_agente.get("pode_responder"))
# print("FASE 2 resposta (tipo):", type(resp_agente.get("resposta")))

# if resp_agente.get("resultado_execucao"):
#     resumo = resp_agente["resultado_execucao"].get("resumo_execucao", {})
#     print(f"Métricas: {resumo.get('metricas_sucesso', 0)} sucesso / {resumo.get('metricas_erro', 0)} erro")
#     erros = resp_agente["resultado_execucao"].get("erros", [])
#     if erros:
#         print("ERROS:", erros)

# print("=" * 60)
# print("\nEntrega final:")
# print(resultado_os["entrega_final"])

### Teste `agente-analise-concessionaria`

Requer `df_os` carregado e `executar_fluxo_maestro` importado da app. Inclua o skill na lista de agentes DataFrame. O Maestro extrai o nome da concessionária da pergunta e injeta filtro + `oss_valor_venda_real > 0` antes do executor.

In [12]:
# --- agente-analise-concessionaria (2 fases) ---
# Ajuste NOME_CONC e PERGUNTA_CONC. Requer df_os no namespace.

NOME_CONC = "Osaka Pampulha"  # exemplo: parte do nome em concessionaria_nome
PERGUNTA_CONC = f"Análise completa da concessionária {NOME_CONC}: faturamento, serviços e vendedoras."

resultado_conc = executar_fluxo_maestro(
    client=client,
    pergunta=PERGUNTA_CONC,
    model=model,
    skills_list=skills,
    agentes=["agente-analise-concessionaria"],
    agentes_dataframe=["agente-analise-concessionaria"],
    verbose=True,
    mysql_host=os.environ.get("MYSQL_HOST"),
    mysql_porta=int(os.environ.get("MYSQL_PORT", "3306")),
    mysql_usuario=os.environ.get("MYSQL_USER"),
    mysql_senha=os.environ.get("MYSQL_PASSWORD"),
    mysql_banco=os.environ.get("MYSQL_DATABASE"),
    mysql_tabelas=[
        {"tabela": "os", "alias": "os", "colunas": [
            "`os`.*",
            "EXISTS(SELECT 1 FROM caixas cx WHERE cx.os_id = os.id AND cx.cancelado = 0 AND cx.deleted_at IS NULL) AS os_paga",
            "(SELECT COUNT(*) FROM os_servicos oss2 WHERE oss2.os_id = os.id) AS qtd_servicos",
        ]},
        {"tabela": "os_servicos", "alias": "oss", "fk": "os.id = oss.os_id", "colunas": [
            "oss.id AS oss_id",
            "oss.codigo AS oss_codigo",
            "oss.valor_venda AS oss_valor_venda",
            "oss.valor_original AS oss_valor_original",
            "oss.desconto_supervisao AS oss_desconto_supervisao",
            "oss.desconto_migracao_cortesia AS oss_desconto_migracao_cortesia",
            "oss.desconto_avista AS oss_desconto_avista",
            "oss.valor_venda_real AS oss_valor_venda_real",
            "oss.desconto_bonus AS oss_desconto_bonus",
            "oss.fechado AS oss_fechado",
            "oss.cancelado AS oss_cancelado",
            "oss.servico_id AS oss_servico_id",
            "oss.os_tipo_id AS oss_os_tipo_id",
            "oss.combo_id AS oss_combo_id",
            "oss.concessionaria_execucao_id AS oss_concessionaria_execucao_id",
            "oss.created_at AS oss_created_at",
            "oss.deleted_at AS oss_deleted_at",
        ]},
        {"tabela": "servicos", "alias": "ser", "fk": "oss.servico_id = ser.id", "colunas": [
            "ser.id AS ser_id",
            "ser.nome AS servico_nome",
            "ser.custo_fixo AS ser_custo_fixo",
            "ser.grupo_servico_id AS ser_grupo_servico_id",
            "ser.subgrupo_servico_id AS ser_subgrupo_servico_id",
            "ser.servico_categoria_id AS ser_servico_categoria_id",
        ]},
        {"tabela": "concessionarias", "alias": "con", "fk": "os.concessionaria_id = con.id", "colunas": [
            "con.id AS con_id",
            "con.nome AS concessionaria_nome",
            "con.uf AS con_uf",
            "con.localidade AS con_localidade",
            "con.cluster_id AS con_cluster_id",
            "con.business_unit_id AS con_business_unit_id",
            "con.gerente_nome AS con_gerente_nome",
        ]},
        {"tabela": "funcionarios", "alias": "func", "fk": "os.vendedor_id = func.id", "colunas": [
            "func.id AS func_id",
            "func.nome AS vendedor_nome",
            "func.terceiros AS func_terceiros",
            "func.funcionario_situacao_id AS func_situacao_id",
        ]},
    ],
    mysql_limite=50_000,
    mysql_filtro_where="os.deleted_at IS NULL AND os.created_at >= '2023-01-01'",
    mysql_injetar_namespace=globals(),
    dataframe_preexistente=None,
)

r0 = (resultado_conc.get("respostas_agentes") or [{}])[0]
print("filtro extraído:", r0.get("filtro_concessionaria_extraido"))
print("entrega (início):\n", (resultado_conc.get("entrega_final") or "")[:1500])

[MAESTRO] Iniciando fluxo. Pergunta: Análise completa da concessionária Osaka Pampulha: faturamento, serviços e vende...
[MAESTRO] Passo 1+2: Analisando pergunta (chamando LLM Maestro)...


2026-03-19 17:32:12,137 [openai._base_client] DEBUG: Request options: {'method': 'post', 'url': '/chat/completions', 'files': None, 'idempotency_key': 'stainless-python-retry-b37c1cf1-e9f5-420b-88e3-13817c148a0a', 'content': None, 'json_data': {'messages': [{'role': 'system', 'content': '---\nname: maestro\nmodel: gpt-5-mini\ndescription: >\n  Orquestrador central de múltiplos agentes especializados. Ative esta skill quando a pergunta do usuário\n  envolver mais de um domínio de conhecimento, ou quando precisar de respostas multidisciplinares com alta\n  precisão. O Maestro analisa a pergunta, seleciona os agentes mais adequados do registro de skills\n  disponíveis, coleta respostas com scores, envia ao Avaliador de Coerência e entrega ao usuário apenas\n  as respostas mais relevantes e consistentes. Use sempre que o usuário mencionar "agentes", "múltiplas\n  perspectivas", "orquestração", ou quando a pergunta cruzar domínios como finanças+tecnologia,\n  saúde+jurídico, dados+negócios,

[MAESTRO] Pré-carregando dados MySQL...
[agente-mysql] Conectando ao banco 'smart'...
[agente-mysql] ✅ 5 tabela(s) validada(s): ['os', 'os_servicos', 'servicos', 'concessionarias', 'funcionarios']
[agente-mysql] Linhas na tabela principal 'os': 173,049
[agente-mysql] Query gerada:
SELECT `os`.*, EXISTS(SELECT 1 FROM caixas cx WHERE cx.os_id = os.id AND cx.cancelado = 0 AND cx.deleted_at IS NULL) AS os_paga, (SELECT COUNT(*) FROM os_servicos oss2 WHERE oss2.os_id = os.id) AS qtd_servicos, oss.id AS oss_id, oss.codigo AS oss_codigo, oss.valor_venda AS oss_valor_venda, oss.valor_original AS oss_valor_original, oss.desconto_supervisao AS oss_desconto_supervisao, oss.desconto_migracao_cortesia AS oss_desconto_migracao_cortesia, oss.desconto_avista AS oss_desconto_avista, oss.valor_venda_real AS oss_valor_venda_real, oss.desconto_bonus AS oss_desconto_bonus, oss.fechado AS oss_fechado, oss.cancelado AS oss_cancelado, oss.servico_id AS oss_servico_id, oss.os_tipo_id AS oss_os_tipo_id, oss.com

/home/cristiano/code/jupter_notebook/app/services/maestro_fluxo.py:351: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0    0
1    0
2    0
3    0
4    0
5    0
6    0
7    1
Name: email_garantia_enviado, dtype: object' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  amostra.iloc[:, i] = amostra.iloc[:, i].astype(object)
/home/cristiano/code/jupter_notebook/app/services/maestro_fluxo.py:361: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0    332946.0
1    332945.0
2    332944.0
3    332942.0
4    332943.0
5    332941.0
6    332940.0
7    332932.0
Name: oss_id, dtype: object' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  amostra.iloc[:, i] = amostra.iloc[:, i].astype(object)
2026-03-19 17:32:41,150 [openai._base_client] DEBUG: Request options: {'method': 'post',

[MAESTRO] DataFrame disponível em 'df_os'.
[MAESTRO] Passo 3: Invocando agente: agente-analise-concessionaria (2 fases)
[MAESTRO]   [agente-analise-concessionaria] FASE 1: solicitando perguntas_dados...


...

[MAESTRO]   [agente-analise-concessionaria] FASE 1 execução ✅: métricas=25
[MAESTRO]   [agente-analise-concessionaria] FASE 2: enviando agregados para interpretação...


2026-03-19 17:36:25,096 [httpcore.http11] DEBUG: receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Thu, 19 Mar 2026 20:36:24 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'CF-Ray', b'9def43440b5dcae0-GIG'), (b'CF-Cache-Status', b'DYNAMIC'), (b'Server', b'cloudflare'), (b'Strict-Transport-Security', b'max-age=31536000; includeSubDomains; preload'), (b'X-Content-Type-Options', b'nosniff'), (b'access-control-expose-headers', b'X-Request-ID'), (b'openai-organization', b'carsoul-pliqq2'), (b'openai-processing-ms', b'66887'), (b'openai-project', b'proj_hxilwBbr6yeydV4YtYGbjIqZ'), (b'openai-version', b'2020-10-01'), (b'x-openai-proxy-wasm', b'v0.1'), (b'x-ratelimit-limit-requests', b'5000'), (b'x-ratelimit-limit-tokens', b'4000000'), (b'x-ratelimit-remaining-requests', b'4999'), (b'x-ratelimit-remaining-tokens', b'3996271'), (b'x-ratelimit-reset-requests', b'12ms'), (b'x-ratelimit-reset-

[MAESTRO] Passo 4: Respostas a enviar ao avaliador: 1 de 1
[MAESTRO] Passo 5: Invocando avaliador de coerência...


2026-03-19 17:36:46,738 [httpcore.http11] DEBUG: receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Thu, 19 Mar 2026 20:36:46 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'CF-Ray', b'9def44e91c36cae0-GIG'), (b'CF-Cache-Status', b'DYNAMIC'), (b'Server', b'cloudflare'), (b'Strict-Transport-Security', b'max-age=31536000; includeSubDomains; preload'), (b'X-Content-Type-Options', b'nosniff'), (b'access-control-expose-headers', b'X-Request-ID'), (b'openai-organization', b'carsoul-pliqq2'), (b'openai-processing-ms', b'20695'), (b'openai-project', b'proj_hxilwBbr6yeydV4YtYGbjIqZ'), (b'openai-version', b'2020-10-01'), (b'x-openai-proxy-wasm', b'v0.1'), (b'x-ratelimit-limit-requests', b'5000'), (b'x-ratelimit-limit-tokens', b'4000000'), (b'x-ratelimit-remaining-requests', b'4999'), (b'x-ratelimit-remaining-tokens', b'3996145'), (b'x-ratelimit-reset-requests', b'12ms'), (b'x-ratelimit-reset-

[MAESTRO] Avaliador retornou ranking: ['agente-analise-concessionaria']
[MAESTRO] Passo 6: Formatando entrega ao usuário.
[MAESTRO] Fluxo concluído.
filtro extraído: {'campo': 'concessionaria_nome', 'valor': 'Osaka Pampulha'}
entrega (início):
 ## Resposta do Maestro

**Pergunta:** Análise completa da concessionária Osaka Pampulha: faturamento, serviços e vendedoras.
**Agentes consultados:** 1 | **Respostas qualificadas:** 1
---
### Análise profunda — concessionária (Osaka Pampulha) — agente-analise-concessionaria
*Score de Confiança: 66%*

**S1 resumo executivo**: Os dados agregados indicam ausência de faturamento registrado (S1_faturamento_total_365d = 0) e zero OS únicas nos últimos 365 dias segundo as métricas retornadas. Séries mensais e diárias vieram vazias ou sem valores úteis. Isso sugere um problema de captura/filtragem (ex.: filtro rígido oss_valor_venda_real>0 sem linhas, ETL que zerou valores, ou dataset errado) em vez de ausência real de atividade. Sem faturamento registr

In [13]:
# =============================================================================
# Sem dataFrame em memória — 2 fases com nova query MySQL
# =============================================================================
# PERGUNTA_TESTE = (
#     "quais concessionárias vendem mais?"
# )

# resultado = executar_fluxo_maestro(
#     client=client,
#     pergunta=PERGUNTA_TESTE,
#     model=model,
#     agentes=["agente-dados", "agente-negocios"],
#     verbose=True,
#     mysql_host=os.environ.get("MYSQL_HOST"),
#     mysql_porta=int(os.environ.get("MYSQL_PORT", "3306")),
#     mysql_usuario=os.environ.get("MYSQL_USER"),
#     mysql_senha=os.environ.get("MYSQL_PASSWORD"),
#     mysql_banco=os.environ.get("MYSQL_DATABASE"),
#     # mysql_tabela="os_servicos",
#     mysql_tabelas=[
#         {"tabela": "os", "alias": "os"},
#         {"tabela": "os_servicos", "alias": "oss", "fk": "os.id = oss.os_id"},
#         {"tabela": "servicos", "alias": "ser", "fk": "oss.servico_id = ser.id", "colunas": ["ser.nome AS servico_nome"]},
#         {"tabela": "concessionarias", "alias": "con", "fk": "os.concessionaria_id = con.id", "colunas": ["con.nome AS concessionaria_nome"]}
#     ],
#     mysql_limite=10_000,
#     mysql_filtro_where="os.deleted_at IS NULL",
#     mysql_injetar_namespace=globals(),
# )
# print(resultado["entrega_final"])

### Maestro com DataFrame já no namespace

Sem `mysql_tabelas`: use `mysql_injetar_namespace=globals()` (ou o dict onde está o DataFrame) e **`dataframe_preexistente="df_os"`** — o Maestro monta `df_contexto` igual ao fluxo MySQL.

**Pré-requisito:** variável `df_os` existir (ex.: célula de carga MySQL acima). Implementação: `app.services.maestro_fluxo.executar_fluxo_maestro`.

In [14]:
# =============================================================================
# DataFrame já em memória — 2 fases sem nova query MySQL
# =============================================================================
# from app.services.maestro_fluxo import executar_fluxo_maestro as executar_fluxo_maestro_app

# _ns = globals()
# if "df_os" not in _ns or _ns["df_os"] is None:
#     raise RuntimeError(
#         "Defina df_os antes (célula MySQL acima, ou df_os = pd.read_parquet(...))."
#     )

# _AGENTES_TESTE = ["agente-analise-os"]
# _AGENTES_DF = ["agente-analise-os"]
# _PERGUNTA = "Resumo executivo: faturamento total válido e top 3 concessionárias por faturamento"

# resultado_so_memoria = executar_fluxo_maestro_app(
#     client=client,
#     pergunta=_PERGUNTA,
#     model=model,
#     skills_list=skills,
#     agentes=_AGENTES_TESTE,
#     agentes_dataframe=_AGENTES_DF,
#     verbose=True,
#     mysql_injetar_namespace=_ns,
#     dataframe_preexistente="df_os",
# )

# print("df_variavel:", resultado_so_memoria.get("df_variavel"))
# print("resultado_mysql:", resultado_so_memoria.get("resultado_mysql"))
# _ef = (resultado_so_memoria.get("respostas_agentes") or [{}])[0].get("resultado_execucao") or {}
# print("métricas:", _ef.get("resumo_execucao"))
# print("\n--- Entrega (início) ---\n")
# print((resultado_so_memoria.get("entrega_final") or "")[:1200])

In [15]:
# =============================================================================
# Teste de geração de GRÁFICOS + PDF a partir do resultado do agente
# =============================================================================
# Pré-requisito: rodar a célula 13 acima (executar_fluxo_maestro)
# Resultado: 8 PNGs em output/graficos/ + 1 PDF de 12 páginas em output/

# import json
# import sys
# from pathlib import Path
# from datetime import datetime, timedelta

# sys.path.insert(0, str(Path(".").resolve()))

# from mnt.skills.agente_analise_os.graficos import gerar_todos_graficos
# from mnt.skills.agente_analise_os.relatorio import gerar_relatorio_pdf

# # --- 1) Recuperar DataFrame da célula anterior ---
# df_var = resultado_os.get("df_variavel")
# if df_var and df_var in globals():
#     df_graficos = globals()[df_var]
# else:
#     raise RuntimeError(f"DataFrame '{df_var}' não encontrado. Rode a célula 13 primeiro.")

# print(f"DataFrame para gráficos: {df_graficos.shape[0]} linhas × {df_graficos.shape[1]} colunas")
# print(f"Colunas necessárias presentes:")
# for col in ["oss_valor_venda_real", "concessionaria_nome", "vendedor_nome", "servico_nome", "os_paga", "qtd_servicos", "created_at"]:
#     presente = col in df_graficos.columns
#     print(f"  {'OK' if presente else 'FALTA'} {col}")

# # --- 2) Gerar os 8 gráficos PNG ---
# print("\n" + "=" * 60)
# print("Gerando gráficos...")
# graficos = gerar_todos_graficos(df_graficos, "output/graficos")
# print(f"\n{len(graficos)} gráficos gerados:")
# for secao, caminho in graficos.items():
#     tamanho = Path(caminho).stat().st_size if Path(caminho).exists() else 0
#     print(f"  {secao}: {caminho} ({tamanho:,} bytes)")

# # --- 3) Extrair análise do agente (FASE 2) ---
# resp_agente = resultado_os["respostas_agentes"][0]
# analise = resp_agente.get("resposta", {})

# if isinstance(analise, str):
#     try:
#         analise = json.loads(analise)
#     except json.JSONDecodeError:
#         print("[AVISO] Resposta do agente não é JSON. Usando como texto do S1.")
#         analise = {"S1_resumo_executivo": analise, "S1_alerta": "normal"}

# # Salvar análise JSON para referência
# Path("output").mkdir(exist_ok=True)
# with open("output/analise_agente.json", "w", encoding="utf-8") as f:
#     json.dump(analise, f, ensure_ascii=False, indent=2)
# print(f"\nAnálise JSON salva em output/analise_agente.json")

# # Mostrar seções presentes
# secoes_esperadas = ["S1_resumo_executivo", "S2_concessionarias", "S3_sazonalidade",
#                     "S4_produtos", "S5_vendedores", "S6_faixas_preco",
#                     "S7_cross_selling", "S8_alertas_anomalias"]
# print("\nSeções da análise:")
# for sec in secoes_esperadas:
#     alerta_key = sec.split("_")[0] + "_alerta"
#     presente = sec in analise
#     nivel = analise.get(alerta_key, "?")
#     preview = str(analise.get(sec, ""))[:80] + "..." if presente else "(ausente)"
#     print(f"  {'OK' if presente else 'FALTA'} {sec} [{nivel}]: {preview}")

# # --- 4) Gerar PDF de 12 páginas ---
# print("\n" + "=" * 60)
# print("Gerando PDF...")

# hoje = datetime.now()
# semana_inicio = (hoje - timedelta(days=hoje.weekday())).strftime("%d/%m/%Y")
# semana_fim = hoje.strftime("%d/%m/%Y")
# periodo = f"{semana_inicio} a {semana_fim}"

# pdf_name = f"relatorio_semanal_os_{hoje.strftime('%Y%m%d')}.pdf"
# pdf_path = f"output/{pdf_name}"

# resultado_pdf = gerar_relatorio_pdf(
#     analise=analise,
#     graficos=graficos,
#     out_path=pdf_path,
#     titulo="Relatório Semanal de Ordens de Serviço",
#     subtitulo="Análise Gerencial Automatizada",
#     periodo=periodo,
# )

# tamanho_pdf = Path(resultado_pdf).stat().st_size
# print(f"\nPDF gerado com sucesso!")
# print(f"  Arquivo: {resultado_pdf}")
# print(f"  Tamanho: {tamanho_pdf:,} bytes ({tamanho_pdf / 1024:.0f} KB)")
# print(f"  Período: {periodo}")

# print("\n" + "=" * 60)
# print("RESUMO:")
# print(f"  Gráficos: {len(graficos)} PNGs em output/graficos/")
# print(f"  PDF: {pdf_path}")
# print(f"  Análise JSON: output/analise_agente.json")
# print("=" * 60)